# System 1 flow core for AlphaEvolve

This notebook is a small, dependency-free extraction of the parts of System 1 that are useful for an evolutionary experiment:

1. the `submission.modality` and `submission.signals` contract,
2. the four modalities,
3. flow sequencing, and
4. the request/prompt used to produce the next activity.

The production flow selector is **not an LLM prompt**. It first chooses a stage with a custom cycle or an independent random draw. Recall and ghost reuse the current card; MCQ and microdrill then call their own generators.

The two functions marked `EVOLVE-BLOCK` are the intended AlphaEvolve mutation surface. Everything else is a stable harness. This notebook does not connect to PostgreSQL or an LLM, so it is safe to run locally.

Production references: `src/practiceFlow.ts`, `src/App.tsx`, `backend/app/models.py`, `backend/app/services/attempts_service.py`, `backend/app/core/generator.py`, and `backend/app/services/micro_drill_service.py`.

In [1]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from enum import Enum
from typing import Any, Mapping, Sequence
import json
import math
import random


## 1. Core data structures

Flow stage names are UI/sequencer names. Modality names are the canonical database values. Keeping the mapping explicit prevents `recall`/`total-recall` and `multiple-choice`/`mcq` drift.

In [2]:
class Modality(str, Enum):
    TOTAL_RECALL = "total-recall"
    GHOST_REP = "ghost-rep"
    MCQ = "mcq"
    MICRODRILL = "microdrill"


class FlowStage(str, Enum):
    RECALL = "recall"
    GHOST = "ghost"
    MULTIPLE_CHOICE = "multiple-choice"
    MICRODRILL = "microdrill"


STAGE_TO_MODALITY = {
    FlowStage.RECALL: Modality.TOTAL_RECALL,
    FlowStage.GHOST: Modality.GHOST_REP,
    FlowStage.MULTIPLE_CHOICE: Modality.MCQ,
    FlowStage.MICRODRILL: Modality.MICRODRILL,
}
MODALITY_TO_STAGE = {modality: stage for stage, modality in STAGE_TO_MODALITY.items()}


@dataclass(frozen=True)
class FlowBlock:
    stage: FlowStage
    count: int


@dataclass(frozen=True)
class FlowConfig:
    mode: str = "custom"  # custom | random
    blocks: tuple[FlowBlock, ...] = (
        FlowBlock(FlowStage.RECALL, 1),
        FlowBlock(FlowStage.GHOST, 2),
        FlowBlock(FlowStage.MULTIPLE_CHOICE, 1),
        FlowBlock(FlowStage.MICRODRILL, 1),
    )
    random_stages: tuple[FlowStage, ...] = tuple(FlowStage)


@dataclass(frozen=True)
class FocusLine:
    line_number: int
    expected: str
    actual: str = ""
    status: str = "missing"  # mismatch | missing | extra


@dataclass(frozen=True)
class Focus:
    summary: str
    missed_lines: tuple[FocusLine, ...] = ()


@dataclass(frozen=True)
class Card:
    card_id: str
    title: str
    algorithm: str
    prompt: str
    target: str
    tags: tuple[str, ...] = ()


@dataclass(frozen=True)
class FlowStep:
    run_id: str
    anchor_card_id: str
    anchor_title: str
    stage: FlowStage
    step: int
    cycle: int
    focus: Focus


## 2. Production-equivalent flow sequencing

`production_next_flow_step` is the direct Python equivalent of `nextFlowStep` in `src/practiceFlow.ts`. The wrapper `choose_next_stage` is deliberately small so AlphaEvolve can replace it with an adaptive policy that reads prior submissions.

In [3]:
def validate_flow_config(config: FlowConfig) -> None:
    if config.mode not in {"custom", "random"}:
        raise ValueError("mode must be 'custom' or 'random'")
    if not config.blocks or any(not 1 <= block.count <= 20 for block in config.blocks):
        raise ValueError("custom flows need blocks with 1-20 reps")
    if not config.random_stages:
        raise ValueError("random flows need at least one enabled stage")


def expand_flow(blocks: Sequence[FlowBlock]) -> tuple[FlowStage, ...]:
    return tuple(stage for block in blocks for stage in [block.stage] * block.count)


def production_next_flow_step(
    config: FlowConfig, step: int, rng: random.Random | None = None
) -> tuple[FlowStage, int]:
    validate_flow_config(config)
    if step < 0:
        raise ValueError("step must be non-negative")
    if config.mode == "random":
        chooser = rng or random.Random()
        return chooser.choice(config.random_stages), 1
    sequence = expand_flow(config.blocks)
    return sequence[step % len(sequence)], step // len(sequence) + 1


# EVOLVE-BLOCK-START: sequencing policy
def choose_next_stage(
    config: FlowConfig,
    step: int,
    history: Sequence[Mapping[str, Any]],
    rng: random.Random | None = None,
) -> tuple[FlowStage, int]:
    """Seed policy: exactly reproduce the production custom/random sequencer."""
    del history  # An evolved policy may use prior modality/evaluation signals.
    return production_next_flow_step(config, step, rng)
# EVOLVE-BLOCK-END


config = FlowConfig()
[choose_next_stage(config, step, [])[0].value for step in range(8)]


['recall',
 'ghost',
 'ghost',
 'multiple-choice',
 'microdrill',
 'recall',
 'ghost',
 'ghost']

## 3. `submission.modality` and `submission.signals`

The production table stores a canonical modality column plus a JSONB signal envelope. The envelope is limited to `elapsed_ms`, `evaluation`, optional `flow`, and `modality`. The nested modality object is the experimental feature surface.

Core PostgreSQL shape (other content/provenance columns omitted here):

```sql
CREATE TABLE submission (
  id BIGSERIAL PRIMARY KEY,
  session_id VARCHAR(80) NOT NULL DEFAULT '0000',
  answer TEXT NOT NULL,
  successful BOOLEAN NOT NULL DEFAULT FALSE,
  signals JSONB NOT NULL,
  modality VARCHAR(30) NOT NULL
    CHECK (modality IN ('total-recall','ghost-rep','mcq','microdrill')),
  created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);
```

In [4]:
ALLOWED_SIGNAL_KEYS = {"elapsed_ms", "evaluation", "flow", "modality"}
BASELINE_MODALITY_FIELDS = {
    Modality.TOTAL_RECALL: {"kind", "exact", "targetLineCount", "submittedLineCount", "missedLineCount"},
    Modality.GHOST_REP: {"kind", "exact", "targetLineCount", "submittedLineCount", "missedLineCount", "focusedLineNumbers"},
    Modality.MCQ: {"kind", "correct", "selectedChoiceId", "correctChoiceId"},
    Modality.MICRODRILL: {"kind", "blankCount", "language", "templateLineCount"},
}


@dataclass(frozen=True)
class Submission:
    session_id: str
    answer: str
    successful: bool
    modality: Modality
    signals: dict[str, Any]
    question_type: str = ""
    correct_answer: str = ""
    generated_card_id: str = ""


def validate_signals(signals: Mapping[str, Any], modality: Modality) -> None:
    extra = set(signals) - ALLOWED_SIGNAL_KEYS
    if extra:
        raise ValueError(f"unsupported top-level signal keys: {sorted(extra)}")
    if not isinstance(signals.get("elapsed_ms"), int) or signals["elapsed_ms"] < 0:
        raise ValueError("elapsed_ms must be a non-negative integer")
    for key in ("evaluation", "flow", "modality"):
        if key in signals and not isinstance(signals[key], dict):
            raise ValueError(f"{key} must be an object")
    if signals.get("modality", {}).get("kind") != modality.value:
        raise ValueError("signals.modality.kind must match submission.modality")


# EVOLVE-BLOCK-START: modality-specific signal capture
def capture_modality_signals(modality: Modality, event: Mapping[str, Any]) -> dict[str, Any]:
    """Seed feature extractor matching the fields currently emitted by System 1."""
    if modality in {Modality.TOTAL_RECALL, Modality.GHOST_REP}:
        features = {
            "exact": bool(event.get("exact", False)),
            "targetLineCount": int(event.get("targetLineCount", 0)),
            "submittedLineCount": int(event.get("submittedLineCount", 0)),
            "missedLineCount": int(event.get("missedLineCount", 0)),
        }
        if modality is Modality.GHOST_REP:
            features["focusedLineNumbers"] = list(event.get("focusedLineNumbers", []))
        return features
    if modality is Modality.MCQ:
        return {
            "correct": bool(event.get("correct", False)),
            "selectedChoiceId": str(event.get("selectedChoiceId", "")),
            "correctChoiceId": str(event.get("correctChoiceId", "")),
        }
    return {
        "blankCount": int(event.get("blankCount", 0)),
        "language": str(event.get("language", "python")),
        "templateLineCount": int(event.get("templateLineCount", 0)),
    }
# EVOLVE-BLOCK-END


def build_submission(
    *,
    flow_step: FlowStep,
    answer: str,
    correct_answer: str,
    successful: bool,
    elapsed_ms: int,
    evaluation: Mapping[str, Any],
    event: Mapping[str, Any],
    question_type: str = "",
) -> Submission:
    modality = STAGE_TO_MODALITY[flow_step.stage]
    modality_signals = {
        **capture_modality_signals(modality, event),
        "kind": modality.value,  # canonical value wins
    }
    signals = {
        "elapsed_ms": int(elapsed_ms),
        "evaluation": dict(evaluation),
        "flow": {
            "runId": flow_step.run_id,
            "anchorCardId": flow_step.anchor_card_id,
            "anchorTitle": flow_step.anchor_title,
            "stage": flow_step.stage.value,
            "step": flow_step.step,
            "cycle": flow_step.cycle,
        },
        "modality": modality_signals,
    }
    validate_signals(signals, modality)
    return Submission(
        session_id=flow_step.run_id,
        answer=answer,
        successful=bool(successful),
        modality=modality,
        signals=signals,
        question_type=question_type,
        correct_answer=correct_answer,
        generated_card_id=flow_step.anchor_card_id,
    )


## 4. Next-activity prompt/request

This dispatcher makes the generation boundary visible. A returned `generator` of `None` means no LLM call is needed. The MCQ and microdrill prompt text mirrors the constraints used by the current backend, while the payload mirrors the flow context assembled by the client.

In [5]:
FLOW_MCQ_SYSTEM_PROMPT = (
    "You generate multiple-choice cards for algorithm pattern recognition and reasoning. "
    'Return only a top-level JSON object shaped exactly like {"drills": [...]}. '
    "Return exactly one drill with id, title, pattern, skill, difficulty, question, choices, "
    "correctChoiceId, explanation, and tags. Choices must contain exactly A, B, C, and D. "
    "Anchor the drill on specimenContext; its prompt and target code are the source of truth. "
    "Test why that exact specimen works: its invariant, state variables, loop or branch decisions, "
    "boundary handling, or likely implementation mistakes. Treat missed lines as the remediation "
    "target. Test why a missed line exists, what bug the drift causes, how to repair it, or what "
    "invariant it protects. Establish the key idea for the next step in a Socratic chain. "
    "Prefer a compact, PEP 8 Python snippet when it makes the idea concrete. Make distractors "
    "plausible adjacent patterns or common code-level mistakes. Return valid JSON only."
)

FLOW_MICRODRILL_SYSTEM_PROMPT = (
    "Create one self-contained coding reinforcement microdrill based on the supplied card and focus. "
    "Vary the scenario for this rep while preserving the core skill. Return strict JSON with prompt "
    "(a short task, no answers), language, template (raw code, no fences), and answers (an ordered "
    "array of strings, one per blank). Use exactly three underscores for each blank and 1-4 blanks. "
    "Each answer must fit on one line. Keep the template under 30 lines. Replacing blanks in order "
    "with answers must produce the correct complete solution. Do not use triple underscores anywhere "
    "else. No explanation field."
)


def build_ghost_target(focus: Focus) -> str:
    return "\n".join(line.expected for line in focus.missed_lines if line.expected.strip()).strip()


def build_next_activity_request(card: Card, flow_step: FlowStep, difficulty: str = "Med.") -> dict[str, Any]:
    stage = flow_step.stage
    if stage is FlowStage.RECALL:
        return {"generator": None, "modality": Modality.TOTAL_RECALL.value, "prompt": card.prompt, "target": card.target}
    if stage is FlowStage.GHOST:
        return {"generator": None, "modality": Modality.GHOST_REP.value, "prompt": card.prompt, "target": build_ghost_target(flow_step.focus)}

    focus_payload = {
        "sequenceStage": stage.value,
        "focusSummary": flow_step.focus.summary,
        "missedLines": [
            {"lineNumber": line.line_number, "expected": line.expected, "actual": line.actual, "status": line.status}
            for line in flow_step.focus.missed_lines
        ],
    }
    if stage is FlowStage.MULTIPLE_CHOICE:
        return {
            "generator": "multiple-choice",
            "modality": Modality.MCQ.value,
            "system_prompt": FLOW_MCQ_SYSTEM_PROMPT,
            "payload": {
                "questionType": f"skill-map-mcq:card:progressive:flow-{flow_step.run_id}-step-{flow_step.step}",
                "count": 1,
                "difficulty": difficulty,
                "sourceMode": "card",
                "flowMode": "progressive",
                "skillMap": [{"algorithm": card.algorithm, "skills": [card.title, *card.tags[:4]]}],
                "specimenContext": {
                    "cardId": card.card_id, "cardTitle": card.title, "algorithm": card.algorithm,
                    "prompt": card.prompt, "target": card.target, "tags": list(card.tags), "focus": focus_payload,
                },
            },
        }
    return {
        "generator": "microdrill",
        "modality": Modality.MICRODRILL.value,
        "system_prompt": FLOW_MICRODRILL_SYSTEM_PROMPT,
        "payload": {
            "cardTitle": card.title, "prompt": card.prompt, "target": card.target,
            "focus": flow_step.focus.summary, "rep": flow_step.step + 1,
        },
    }


## 5. Tiny end-to-end example

The synthetic episode below exercises every modality. Replace `episode` with rows exported from PostgreSQL using:

```sql
SELECT session_id, answer, successful, modality, signals, question_type, correct_answer, generated_card_id
FROM submission
WHERE signals ? 'flow'
ORDER BY signals->'flow'->>'runId', (signals->'flow'->>'step')::int;
```

In [6]:
card = Card(
    card_id="sliding-window",
    title="Variable Sliding Window",
    algorithm="Sliding Window",
    prompt="Write the reusable longest-valid-window skeleton.",
    target=(
        "left = 0\n"
        "for right, value in enumerate(values):\n"
        "    add(value)\n"
        "    while not valid():\n"
        "        remove(values[left])\n"
        "        left += 1\n"
        "    best = max(best, right - left + 1)"
    ),
    tags=("sliding-window", "expand-shrink"),
)
focus = Focus(
    summary="Target the shrink loop and left-boundary update.",
    missed_lines=(
        FocusLine(4, "    while not valid():", "    if not valid():", "mismatch"),
        FocusLine(6, "        left += 1", "", "missing"),
    ),
)

sample_events = {
    FlowStage.RECALL: {"exact": False, "targetLineCount": 7, "submittedLineCount": 6, "missedLineCount": 2},
    FlowStage.GHOST: {"exact": True, "targetLineCount": 2, "submittedLineCount": 2, "missedLineCount": 0, "focusedLineNumbers": [4, 6]},
    FlowStage.MULTIPLE_CHOICE: {"correct": True, "selectedChoiceId": "B", "correctChoiceId": "B"},
    FlowStage.MICRODRILL: {"blankCount": 2, "language": "python", "templateLineCount": 7},
}
sample_scores = [42.0, 64.0, 82.0, 91.0]
episode: list[Submission] = []
for step, stage in enumerate(FlowStage):
    flow_step = FlowStep("demo-run", card.card_id, card.title, stage, step, 1, focus)
    episode.append(build_submission(
        flow_step=flow_step, answer="demo", correct_answer="demo",
        successful=sample_scores[step] >= 80, elapsed_ms=30_000 - step * 4_000,
        evaluation={"version": 1, "verdict": "sound" if sample_scores[step] >= 80 else "needs-work", "score": {"overall": sample_scores[step]}},
        event=sample_events[stage], question_type="skill-map",
    ))

print(json.dumps(asdict(episode[0]), indent=2, default=str))
print("\nMCQ request generator:", build_next_activity_request(
    card, FlowStep("demo-run", card.card_id, card.title, FlowStage.MULTIPLE_CHOICE, 2, 1, focus)
)["generator"])


{
  "session_id": "demo-run",
  "answer": "demo",
  "successful": false,
  "modality": "total-recall",
  "signals": {
    "elapsed_ms": 30000,
    "evaluation": {
      "version": 1,
      "verdict": "needs-work",
      "score": {
        "overall": 42.0
      }
    },
    "flow": {
      "runId": "demo-run",
      "anchorCardId": "sliding-window",
      "anchorTitle": "Variable Sliding Window",
      "stage": "recall",
      "step": 0,
      "cycle": 1
    },
    "modality": {
      "exact": false,
      "targetLineCount": 7,
      "submittedLineCount": 6,
      "missedLineCount": 2,
      "kind": "total-recall"
    }
  },
  "question_type": "skill-map",
  "correct_answer": "demo",
  "generated_card_id": "sliding-window"
}

MCQ request generator: multiple-choice


## 6. Small numeric evaluator for evolution

AlphaEvolve needs objective, repeatable scores. This starter evaluator rewards measured learning gain, successful outcomes, complete instrumentation, and modality coverage, with a small time cost. It scores a completed episode produced by a candidate policy; it does **not** claim that historical replay can estimate the counterfactual effect of an unseen policy. Use a simulator, shadow traffic, or a controlled online experiment to produce candidate episodes.

Keep the evaluator fixed while evolving the two mutation blocks. Change the weights only when the product objective changes.

In [7]:
def overall_score(submission: Submission) -> float:
    raw = submission.signals.get("evaluation", {}).get("score", {}).get("overall", 0.0)
    try:
        return max(0.0, min(100.0, float(raw)))
    except (TypeError, ValueError):
        return 0.0


def signal_completeness(submission: Submission) -> float:
    actual = set(submission.signals.get("modality", {}))
    expected = BASELINE_MODALITY_FIELDS[submission.modality]
    return len(actual & expected) / len(expected)


def evaluate_episode(submissions: Sequence[Submission]) -> dict[str, float]:
    if not submissions:
        return {"fitness": 0.0, "learning_gain": 0.0, "success_rate": 0.0, "signal_completeness": 0.0, "coverage": 0.0, "time_cost": 1.0}
    scores = [overall_score(item) for item in submissions]
    learning_gain = max(-1.0, min(1.0, (scores[-1] - scores[0]) / 100.0))
    success_rate = sum(item.successful for item in submissions) / len(submissions)
    completeness = sum(signal_completeness(item) for item in submissions) / len(submissions)
    coverage = len({item.modality for item in submissions}) / len(Modality)
    mean_seconds = sum(item.signals["elapsed_ms"] for item in submissions) / len(submissions) / 1000.0
    time_cost = min(1.0, math.log1p(mean_seconds) / math.log1p(120.0))
    fitness = 0.45 * learning_gain + 0.25 * success_rate + 0.15 * completeness + 0.10 * coverage - 0.05 * time_cost
    return {
        "fitness": round(fitness, 6),
        "learning_gain": round(learning_gain, 6),
        "success_rate": round(success_rate, 6),
        "signal_completeness": round(completeness, 6),
        "coverage": round(coverage, 6),
        "time_cost": round(time_cost, 6),
    }


metrics = evaluate_episode(episode)
metrics


{'fitness': 0.561941,
 'learning_gain': 0.49,
 'success_rate': 0.5,
 'signal_completeness': 1.0,
 'coverage': 1.0,
 'time_cost': 0.671188}

## 7. Experiment contract

For an AlphaEvolve run, extract the code cells into a seed `.py` program and expose an evaluator that returns `evaluate_episode(...)["fitness"]` plus the component metrics. Candidate code should be allowed to modify only:

- `capture_modality_signals`: add low-cost, outcome-relevant fields inside `signals.modality`; and
- `choose_next_stage`: choose among the four allowed stages from prior submissions.

Hard constraints worth retaining in the evaluator:

- always return one of the four stages,
- keep `signals` JSON-serializable and within the four top-level keys,
- force `signals.modality.kind == submission.modality`,
- cap episode length and LLM calls, and
- evaluate on held-out learners/cards to prevent a policy that memorizes the benchmark.

The notebook intentionally stops before calling AlphaEvolve: credentials, the candidate execution sandbox, and the source of outcome-bearing episodes are deployment choices.